# 04 · Análisis

Cuadrantes con las **medianas** del grupo como cortes, intervalos de incertidumbre para
ambos ejes, dispersión de precios por producto y comparación de cadenas. Ninguna
relación se interpreta como causal.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PROC = ROOT / "data/processed"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
from src.basket import bootstrap_basket, dispersion, relative_prices, CANDIDATOS
import warnings
params = json.loads((ROOT / "data/metadata/canasta_parametros.json").read_text(encoding="utf-8"))
items = params["productos"]
names = CANDIDATOS.set_index("producto_normalizado").nombre
final = pd.read_csv(PROC / "dataset_final.csv", dtype={"cd_ensu": str})
prices = pd.read_csv(PROC / "precios_area_producto.csv", dtype={"cd_ensu": str})
obs = pd.read_csv(PROC / "observaciones_canasta.csv", dtype={"cd_ensu": str})
metodos = pd.read_csv(PROC / "comparacion_metodos_canasta.csv")
areas = final.area_urbana_ensu.tolist()
print(len(final), "áreas")

44 áreas


## 1. Incertidumbre del costo

Bootstrap con 1,000 réplicas, remuestreando establecimientos dentro de cada área, que es
la fuente principal de variación (≈5 supermercados por área).

In [2]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    boot = bootstrap_basket(obs, items, areas, reps=1000)
res = final.merge(boot, on="area_urbana_ensu")
res["ancho_ic_pct"] = (res.costo_ic90_sup - res.costo_ic90_inf) / res.costo_canasta * 100
rango = res.costo_canasta.max() / res.costo_canasta.min() - 1
print(f"Rango del costo: ${res.costo_canasta.min():,.0f} – ${res.costo_canasta.max():,.0f} ({rango:.1%})")
print(f"Ancho mediano del IC90 por área: {res.ancho_ic_pct.median():.1f}% del costo")
display(res[["costo_canasta", "ancho_ic_pct", "seguridad_pct"]].describe().round(1))

Rango del costo: $944 – $1,068 (13.1%)
Ancho mediano del IC90 por área: 6.3% del costo


,costo_canasta,ancho_ic_pct,seguridad_pct
count,44.0,44.0,44.0
mean,1006.2,6.6,39.6
std,21.1,2.7,18.3
min,944.3,2.1,9.2
25%,993.7,4.9,26.9
50%,1007.2,6.3,34.7
75%,1017.9,7.2,55.7
max,1067.8,15.3,83.1


## 2. Cuadrantes

In [3]:
x_med, y_med = res.costo_canasta.median(), res.seguridad_pct.median()
barato, seguro = res.costo_canasta <= x_med, res.seguridad_pct >= y_med
res["cuadrante"] = np.select([barato & seguro, ~barato & seguro, barato & ~seguro], [1, 2, 3], 4)
res["cuadrante_nombre"] = res.cuadrante.map({1: "Menor costo + mayor seguridad", 2: "Mayor costo + mayor seguridad",
                                             3: "Menor costo + menor seguridad", 4: "Mayor costo + menor seguridad"})
# Robusto: los IC90 de ambos ejes quedan del mismo lado de su mediana.
x_ok = np.where(barato, res.costo_ic90_sup <= x_med, res.costo_ic90_inf > x_med)
y_ok = np.where(seguro, res.ic90_inf_seguro >= y_med, res.ic90_sup_seguro < y_med)
res["cuadrante_robusto"] = x_ok & y_ok
res["eje_costo_robusto"], res["eje_seguridad_robusto"] = x_ok, y_ok
print(f"Mediana costo ${x_med:,.2f} · mediana seguridad {y_med:.1f}%")
display(pd.crosstab(res.cuadrante_nombre, res.cuadrante_robusto, margins=True))
print("Ejes robustos — costo:", res.eje_costo_robusto.sum(), "· seguridad:", res.eje_seguridad_robusto.sum())
for q in [1, 2, 3, 4]:
    d = res[res.cuadrante == q].sort_values("costo_canasta")
    display(Markdown(f"**{d.cuadrante_nombre.iloc[0]}** ({len(d)}): " +
        ", ".join(f"{a.rstrip('0123456789')}{'*' if r else ''}" for a, r in zip(d.area_urbana_ensu, d.cuadrante_robusto))))
display(Markdown("\\* cuadrante robusto en ambos ejes."))

Mediana costo $1,007.25 · mediana seguridad 34.7%


cuadrante_robusto,False,True,All
cuadrante_nombre,,,
Mayor costo + mayor seguridad,10,1,11
Mayor costo + menor seguridad,11,0,11
Menor costo + mayor seguridad,10,1,11
Menor costo + menor seguridad,10,1,11
All,41,3,44


Ejes robustos — costo: 5 · seguridad: 30


**Menor costo + mayor seguridad** (11): Ciudad Nezahualcóyotl*, La Paz, Aguascalientes, Atizapán de Zaragoza, Venustiano Carranza, Benito Juárez, Álvaro Obregón, Veracruz, Miguel Hidalgo, Hermosillo, Juárez

**Mayor costo + mayor seguridad** (11): Coyoacán, Querétaro, Tampico, Tlalpan, Cuauhtémoc, Saltillo, Mérida, Durango, Azcapotzalco, San Luis Potosí, Monterrey*

**Menor costo + menor seguridad** (11): Ecatepec de Morelos*, Acapulco de Juárez, San Francisco de Campeche, Tijuana, Cuernavaca, Iztapalapa, Tuxtla Gutiérrez, Oaxaca de Juárez, León de los Aldama, Tlalnepantla de Baz, Heroica Puebla de Zaragoza

**Mayor costo + menor seguridad** (11): Reynosa, Culiacán Rosales, Cuautitlán Izcalli, Villahermosa, Cancún, Gustavo A. Madero, Naucalpan de Juárez, Toluca de Lerdo, Morelia, Zacatecas, Guadalajara

\* cuadrante robusto en ambos ejes.

In [4]:
rho = res[["costo_canasta", "seguridad_pct"]].corr(method="spearman").iloc[0, 1]
pear = res[["costo_canasta", "seguridad_pct"]].corr().iloc[0, 1]
# Permutación: ¿qué tan frecuente es un |rho| así si no hubiera relación?
rng = np.random.default_rng(1)
xr, yr = res.costo_canasta.rank().to_numpy(), res.seguridad_pct.rank().to_numpy()
perm = np.array([np.corrcoef(xr, rng.permutation(yr))[0, 1] for _ in range(5000)])
p_perm = np.mean(np.abs(perm) >= abs(rho))
print(f"Spearman {rho:+.3f} · Pearson {pear:+.3f} · p (permutación) {p_perm:.2f}")

Spearman -0.067 · Pearson -0.114 · p (permutación) 0.67


## 3. Sensibilidad

¿Cambia el cuadrante de costo si se usa la media o una sola marca por producto?

In [5]:
sens = res[["area_urbana_ensu", "costo_canasta"]].merge(metodos, on="area_urbana_ensu", how="left")
for col in ["A_media", "B_misma_marca"]:
    ok = sens[col].notna()
    same = ((sens.loc[ok, col] <= sens.loc[ok, col].median()) == (sens.loc[ok, "costo_canasta"] <= x_med)).mean()
    print(f"{col}: {ok.sum()} áreas; misma mitad de costo que A-mediana en {same:.0%}; "
          f"Spearman {sens.loc[ok, ['costo_canasta', col]].corr(method='spearman').iloc[0, 1]:.2f}")
sin_moderada = res[res.precision_ensu == "alta"]
print(f"Solo precisión ENSU alta ({len(sin_moderada)} áreas): Spearman "
      f"{sin_moderada[['costo_canasta', 'seguridad_pct']].corr(method='spearman').iloc[0, 1]:+.3f}")

A_media: 44 áreas; misma mitad de costo que A-mediana en 77%; Spearman 0.81
B_misma_marca: 42 áreas; misma mitad de costo que A-mediana en 79%; Spearman 0.83
Solo precisión ENSU alta (40 áreas): Spearman -0.039


## 4. ¿Qué productos explican las diferencias?

In [6]:
p = prices[prices.area_urbana_ensu.isin(areas) & prices.producto_normalizado.isin(items)]
disp = dispersion(p, items)
qty = CANDIDATOS.set_index("producto_normalizado").cantidad_canasta
p = p.assign(costo_item=p.precio_mediana * p.producto_normalizado.map(qty))
share = p.groupby("producto_normalizado").costo_item.median() / p.groupby("area_urbana_ensu").costo_item.sum().median() * 100
contrib = p.groupby("producto_normalizado").costo_item.std() ** 2
disp["peso_en_canasta_pct"] = share
disp["contribucion_varianza_pct"] = contrib / contrib.sum() * 100  # aproximada: ignora covarianzas
disp.insert(0, "nombre", names.reindex(disp.index))
display(disp.round(2))
rel = relative_prices(p, items)

,nombre,areas,mediana,minimo,maximo,cv_pct,razon_max_min,peso_en_canasta_pct,contribucion_varianza_pct
producto_normalizado,,,,,,,,,
frijol_negro,Frijol negro,44,25.56,21.68,49.22,21.12,2.27,2.54,10.15
lenteja,Lenteja,44,42.00,32.20,63.90,18.54,1.98,4.17,18.41
arroz,Arroz súper extra,44,24.97,20.40,38.33,16.92,1.88,2.48,5.44
jitomate,Jitomate saladet,44,16.06,9.90,21.72,15.78,2.19,1.59,1.94
cebolla,Cebolla blanca,44,45.95,32.05,62.00,14.93,1.93,4.56,14.27
papa,Papa blanca,44,50.95,44.85,66.00,12.15,1.47,5.06,11.64
frijol_pinto,Frijol pinto,44,33.28,30.18,44.33,10.39,1.47,3.30,3.58
platano,Plátano tabasco,44,19.92,17.45,25.00,9.46,1.43,1.98,1.07
cerdo_chuleta,Chuleta de cerdo,44,104.00,74.55,119.00,8.12,1.60,10.33,19.60


## 5. Comparación de cadenas

Para comparar cadenas con justicia se compara **el mismo producto en la misma área**:
índice = precio del establecimiento / mediana del producto en el área × 100. Solo
cadenas presentes en al menos 5 áreas del dataset final.

In [7]:
st = (obs[obs.area_urbana_ensu.isin(areas) & obs.producto_normalizado.isin(items)]
      .groupby(["area_urbana_ensu", "cadena_comercial", "establecimiento_id", "producto_normalizado"])
      .precio_unitario.median().reset_index())
st["indice"] = st.precio_unitario / st.groupby(["area_urbana_ensu", "producto_normalizado"]).precio_unitario.transform("median") * 100
chains = st.groupby("cadena_comercial").agg(indice_medio=("indice", "mean"), indice_mediano=("indice", "median"), areas=("area_urbana_ensu", "nunique"),
                                            establecimientos=("establecimiento_id", "nunique"), precios=("indice", "size"))
chains = chains[chains.areas >= 5].sort_values("indice_medio").reset_index()
# La mediana del índice suele ser 100 porque muchas tiendas comparten precio exacto;
# la media sí refleja diferencias sistemáticas.
chains["pct_precios_menores_a_mediana"] = chains.cadena_comercial.map(
    st.assign(menor=st.indice < 100).groupby("cadena_comercial").menor.mean() * 100)
display(chains.round(1))

,cadena_comercial,indice_medio,indice_mediano,areas,establecimientos,precios,pct_precios_menores_a_mediana
0,Bodega Aurrera Express,96.2,100.0,6,7,84,47.6
1,Chedraui,97.6,98.7,21,21,419,55.1
2,Bodega Aurrera,98.3,98.8,36,46,869,57.7
3,Mercado Soriana,100.0,100.0,6,8,152,49.3
4,Wal-mart,100.7,100.0,39,47,948,31.9
5,Mega Soriana,102.3,100.0,18,19,379,37.2
6,Soriana Super,102.9,100.0,8,8,161,39.8
7,Superissste,102.9,100.0,6,10,137,43.1
8,Hipermercado Soriana,103.0,100.0,35,42,848,38.1
9,Wal-mart Express,103.9,100.2,12,18,345,17.4


In [8]:
res.to_csv(PROC / "resultados_analisis.csv", index=False, encoding="utf-8-sig")
disp.reset_index().to_csv(PROC / "dispersion_productos.csv", index=False, encoding="utf-8-sig")
rel.to_csv(PROC / "precios_relativos_area_producto.csv", encoding="utf-8-sig")
chains.to_csv(PROC / "comparacion_cadenas.csv", index=False, encoding="utf-8-sig")
meta = {"areas": len(res), "productos": len(items), "mediana_costo": round(float(x_med), 2),
        "mediana_seguridad": round(float(y_med), 2), "rango_costo_pct": round(rango * 100, 1),
        "spearman": round(float(rho), 3), "p_permutacion": round(float(p_perm), 3),
        "periodo_precios": "16–31 ago 2026", "periodo_ensu": "junio 2026",
        "fecha_inicio": params["fecha_inicio"], "fecha_fin": params["fecha_fin"]}
(ROOT / "data/metadata/resultados_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
top = disp.index[0]
q1 = res[(res.cuadrante == 1) & res.cuadrante_robusto].area_urbana_ensu.str.rstrip("0123456789").tolist()
q4 = res[(res.cuadrante == 4) & res.cuadrante_robusto].area_urbana_ensu.str.rstrip("0123456789").tolist()
display(Markdown(f"""**Hallazgos**

- La canasta de {len(items)} productos cuesta entre **${res.costo_canasta.min():,.0f}** y **${res.costo_canasta.max():,.0f}**:
  una diferencia de {rango:.0%}. El intervalo de 90 % de cada área mide en mediana {res.ancho_ic_pct.median():.0f}% del costo,
  así que buena parte del ordenamiento por costo no es concluyente.
- La seguridad percibida varía mucho más: de {res.seguridad_pct.min():.0f}% a {res.seguridad_pct.max():.0f}%.
- No se observa relación entre ambas dimensiones (Spearman {rho:+.2f}; p por permutación {p_perm:.2f}).
- Cuadrante robusto «menor costo + mayor seguridad»: {', '.join(q1) or 'ninguna'}.
  Cuadrante robusto «mayor costo + menor seguridad»: {', '.join(q4) or 'ninguna'}.
- El producto con mayor dispersión geográfica es **{names[top]}** (CV {disp.cv_pct.iloc[0]:.0f}%).
"""))

**Hallazgos**

- La canasta de 21 productos cuesta entre **$944** y **$1,068**:
  una diferencia de 13%. El intervalo de 90 % de cada área mide en mediana 6% del costo,
  así que buena parte del ordenamiento por costo no es concluyente.
- La seguridad percibida varía mucho más: de 9% a 83%.
- No se observa relación entre ambas dimensiones (Spearman -0.07; p por permutación 0.67).
- Cuadrante robusto «menor costo + mayor seguridad»: Ciudad Nezahualcóyotl.
  Cuadrante robusto «mayor costo + menor seguridad»: ninguna.
- El producto con mayor dispersión geográfica es **Frijol negro** (CV 21%).
